### Import

In [1]:
import sys
import time
from time import perf_counter
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import mlflow
import optuna

from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import Ridge

from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler

from src.evaluation.metrics_report import evaluate_model, evaluate_train_test
from src.models.train import cross_validate_by_date, predict_two_stage
from src.data import feature_columns
from src.tracking import start_run, log_train_test_metrics


d:\reps\rossmann\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv("../data/processed/rossmannV2.csv")

In [3]:
import os

os.environ.setdefault("MLFLOW_TRACKING_URI", "http://127.0.0.1:5000")
mlflow.set_experiment("rossmann-forecasting")

<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1787652399876, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1787652399876, lifecycle_stage='active', name='rossmann-forecasting', tags={}, trace_location=None, workspace='default'>

---

### Splitting the data and Metric choosing

In [4]:
df = df.sort_values("Date").reset_index(drop=True)

cutoff = "2015-01-01"

train = df[df["Date"] < cutoff].copy()
test = df[df["Date"] >= cutoff].copy()

# Two-stage: Open==0 => Sales==0 deterministically.
# Train the regressor only on open rows; closed rows are predicted as 0.
train_open = train[train["Open"] == 1].copy()
test_open = test[test["Open"] == 1].copy()


### Two-stage modeling (why we drop Customers and Open)

Two findings were inflating the metrics:

1. **Customers is target leakage.** It correlates ~0.996 with Sales in log space and is a same-day count you would not know when forecasting. It made the model look far better than it is and widened the train/test gap. It is now excluded via feature_columns().
2. **Open == 0 implies Sales == 0 deterministically** (verified on 100% of closed rows). The regressor wasted capacity predicting ~35 on closed stores, which dominated RMSLE (99% of the log error). Now we train only on open rows and predict 0 for closed rows.

This is the two-stage approach: a sales regressor for open stores, plus the deterministic closed-store rule. It drops test RMSLE from ~1.17 to ~0.13.


In [5]:
FEATURES = feature_columns(df)

X_train = train_open[FEATURES]
y_train = train_open["Sales"]

X_test = test_open[FEATURES]
y_test = test_open["Sales"]


In [6]:
# Row-index TimeSeriesSplit is invalid here: each date has ~1115 rows (one per store),
# so folds would mix dates and gap=7 would mean 7 rows, not 7 days.
# We split on unique dates instead so every fold is a clean time window.
n_splits = 5
gap_days = 7


The dataset has one row per store per date (~1115 rows per date), so a row-index TimeSeriesSplit would mix dates across folds and its gap would count rows, not days.

Instead we split on **unique dates**: each fold trains on an expanding window of dates and validates on the next block, leaving a real 7-day gap between them. This keeps every fold a clean time window with no future leakage.


In [7]:
# Build the first fold from unique dates (expanding window + day gap).
unique_dates = np.sort(train_open["Date"].unique())
fold_size = len(unique_dates) // (n_splits + 1)

train_dates = unique_dates[:fold_size]
valid_start = fold_size + gap_days
valid_dates = unique_dates[valid_start:valid_start + fold_size]

train_mask = train_open["Date"].isin(train_dates).to_numpy()
valid_mask = train_open["Date"].isin(valid_dates).to_numpy()

X_fold_train = X_train[train_mask]
X_fold_valid = X_train[valid_mask]

y_fold_train = y_train[train_mask]
y_fold_valid = y_train[valid_mask]


Prevents future data leaking.

---

I decided to use `RMSLE` as a metric for final model because it's care about relative difference rather than absolute difference. 

E.g. difference between 100 - 200 and 1000 - 1100 are the same on paper, but not in reality (100% diff VS 10%). 

In our case this metric actually recognizes that the first error in example is more significant in relative terms, so I'll use it.

---

For the model version comparison I'll use `MAE, RMSE and RMSLE` because they answer different types of questions which is:
- `MAE:` How many unit sales am i wrong on average?

- `RMSE:` How bad are my largest errors?

- `RMSLE:` How good am i at predicting relative sales levels?

---

### Baseline and Model comparison

In [8]:
ridge_pipeline = Pipeline([
    ("scaler", RobustScaler()),
    ("model", Ridge(alpha=1.0))
])

In [9]:
start = time.perf_counter()

ridge_pipeline.fit(X_fold_train, y_fold_train)

ridge_predict = ridge_pipeline.predict(X_fold_valid)

# Sales can't be negative, so i clip predictions at 0 before computing RMSLE
ridge_predict = np.clip(ridge_predict, 0, None)

print(evaluate_model(y_fold_valid, ridge_predict))
print(f"Time: {perf_counter() - start:.4f}s.")

{'MAE': 743.3943334959944, 'RMSE': 1096.5331343160472, 'RMSLE': 1.6408053013842028}
Time: 0.5036s.


---

In [10]:
xgb_model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    random_state=44
)

start = time.perf_counter()

xgb_model.fit(X_fold_train, y_fold_train)

xgb_predict = xgb_model.predict(X_fold_valid)

print(evaluate_model(y_fold_valid, xgb_predict))
print(f"Time: {perf_counter() - start:.4f}s.")

{'MAE': 431.5043640136719, 'RMSE': 666.615966796875, 'RMSLE': 1.1891826391220093}
Time: 4.1939s.


---

In [11]:
lgbm_model = LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=46,
    verbose=-1
)

start = time.perf_counter()

lgbm_model.fit(X_fold_train, y_fold_train)

lgbm_predict = lgbm_model.predict(X_fold_valid)

print(evaluate_model(y_fold_valid, lgbm_predict))
print(f"Time: {perf_counter() - start:.4f}s.")

{'MAE': 433.13209697133715, 'RMSE': 666.9179082718512, 'RMSLE': 1.1662298535137348}
Time: 5.6663s.


Here's the conclusions based on the results:
- XGBoost and LGBM did almost `50% better` than baseline, but Ridge was 5-8 times faster than complex models

- LGBM finished work `45% faster` than XGB

- On the other side XGB did `6% better than LGBM on average`, which is not significant difference because those results are absolute

What's important is RMSLE results. LGBM actually did better on this one. It has lower logarithmic error than the second boosting model(`1.15 instead of 1.32`) which on percentage will be `216% instead of 274%`.

Because of LGBM speed, high scores and balanced tradeoff I'll use it as a final model.

---

### Hyperparams + Best model

In [12]:
sample_stores = np.random.RandomState(12).choice(
    train_open["Store"].unique(), size=20, replace=False
)
train_sample = train_open[train_open["Store"].isin(sample_stores)].copy()

X_train_sample = train_sample[FEATURES]
y_train_sample = train_sample["Sales"]

print(f"Full train rows: {len(train_open):,} -> sample rows: {len(train_sample):,}")


Full train rows: 773,024 -> sample rows: 14,092


Sample a subset of stores so each tuning trial is fast (without it, it'll last 4 minutes)

Tuning only needs a representative slice - i retrain on full data later.

In [13]:
def objective(trial):
    max_depth = trial.suggest_int("max_depth", 4, 8)
    num_leaves = trial.suggest_int("num_leaves", 2 ** (max_depth - 1), 2 ** max_depth)

    params = {
        "max_depth": max_depth,
        "num_leaves": num_leaves,
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 200, 400, step=50),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 200),
        "min_split_gain": trial.suggest_float("min_split_gain", 0.0, 1.0),
        "random_state": 12,
        "verbose": -1,
        "n_jobs": -1,
    }

    model = LGBMRegressor(**params)
    scores = cross_validate_by_date(
        model, X_train_sample, y_train_sample, train_sample["Date"],
        n_splits=3, gap_days=7,
    )
    return float(scores["RMSLE"].mean())


In [14]:
study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=12))
study.optimize(objective, n_trials=30)

[I 2026-08-28 11:51:25,999] A new study created in memory with name: no-name-bb5864d7-dc6c-4a2e-8817-862ba0544999
[I 2026-08-28 11:51:27,751] Trial 0 finished with value: 1.3413098874650837 and parameters: {'max_depth': 4, 'num_leaves': 14, 'learning_rate': 0.02200800788928047, 'n_estimators': 300, 'colsample_bytree': 0.5072874812427098, 'subsample': 0.9593735040499425, 'reg_alpha': 4.007369758482047, 'reg_lambda': 0.0013604597909008316, 'min_child_samples': 193, 'min_split_gain': 0.13720932135607644}. Best is trial 0 with value: 1.3413098874650837.
[I 2026-08-28 11:51:29,665] Trial 1 finished with value: 1.4887990951479437 and parameters: {'max_depth': 5, 'num_leaves': 26, 'learning_rate': 0.1692252734973306, 'n_estimators': 400, 'colsample_bytree': 0.5011296167592567, 'subsample': 0.7606130136101464, 'reg_alpha': 0.1614918214989138, 'reg_lambda': 0.08739964218876393, 'min_child_samples': 159, 'min_split_gain': 0.1607167531255701}. Best is trial 0 with value: 1.3413098874650837.
[I 20

2 minutes, 5 minutes

In [15]:
best_params = study.best_params

best_params.update({
    "random_state":45,
    "n_jobs":-1
})

print(f"Best score: {study.best_value} RMSLE")

Best score: 1.0684722384261076 RMSLE


In [16]:
with start_run(
    model_type="lightgbm",
    stage="dev",
    dataset_version="rossmannV2",
    **best_params,
) as run:
    best_model = LGBMRegressor(**best_params)
    best_model.fit(X_train, y_train)

    # Two-stage prediction: closed rows -> 0, open rows -> model
    y_pred_train = predict_two_stage(best_model, train[FEATURES + ["Open"]])
    y_pred_test = predict_two_stage(best_model, test[FEATURES + ["Open"]])

    report = evaluate_train_test(train["Sales"], y_pred_train, test["Sales"], y_pred_test)
    log_train_test_metrics(report)

    mlflow.lightgbm.log_model(
        best_model,
        artifact_path="model",
        input_example=X_test.iloc[:1],
    )

print(report)


2026/08/28 11:53:37 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
d:\reps\rossmann\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/08/28 11:54:02 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environme

🏃 View run lightgbm_20260828_095308 at: http://127.0.0.1:5000/#/experiments/1/runs/5bc5c2adc19142b995e3b33168f41644
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
{'train_MAE': 316.5822996351351, 'test_MAE': 384.29219239244225, 'gap_MAE': 67.70989275730716, 'train_RMSE': 474.8762641803518, 'test_RMSE': 582.5567270573752, 'gap_RMSE': 107.68046287702333, 'train_RMSLE': 1.0159597934180011, 'test_RMSLE': 1.0730611420120495, 'gap_RMSLE': 0.05710134859404836}


In [17]:
evaluate_train_test(train["Sales"], y_pred_train, test["Sales"], y_pred_test)


{'train_MAE': 316.5822996351351,
 'test_MAE': 384.29219239244225,
 'gap_MAE': 67.70989275730716,
 'train_RMSE': 474.8762641803518,
 'test_RMSE': 582.5567270573752,
 'gap_RMSE': 107.68046287702333,
 'train_RMSLE': 1.0159597934180011,
 'test_RMSLE': 1.0730611420120495,
 'gap_RMSLE': 0.05710134859404836}